In [ ]:
import easyocr
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FileUpload, Output, VBox, HBox, Button, Label, FloatSlider
from IPython.display import display
from PIL import Image
import io

#OCR (Optical Character Recognition) 
reader = easyocr.Reader(['en'])  # Default menggunakan bahasa Inggris
output = Output()
upload_btn = FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
process_btn = Button(description="Deteksi Plat Nomor", button_style='info', layout={'width': '200px'})
reset_btn = Button(description="Reset", button_style='danger', layout={'width': '200px'})
download_btn = Button(description="Download Hasil OCR", button_style='primary', layout={'width': '200px'})
show_original_btn = Button(description="Tampilkan Gambar Asli", button_style='warning', layout={'width': '200px'})
uploaded_image = {'data': None}

# Slider untuk mengatur kontras gambar
contrast_slider = FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Kontras:', continuous_update=False)

# Fungsi untuk menyesuaikan kontras gambar
def adjust_contrast(image, contrast_factor):
    # Konversi gambar ke tipe yang lebih dapat diproses oleh OpenCV
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Meningkatkan kontras dengan mengalikan saluran L (luminositas)
    l = cv2.convertScaleAbs(l, alpha=contrast_factor, beta=0)
    lab = cv2.merge([l, a, b])

    # Kembalikan gambar ke RGB
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

# Fungsi bantu untuk menghitung y tengah dari bbox
def y_center(bbox):
    return np.mean([point[1] for point in bbox])

# Upload handler
def on_upload_change(change):
    if upload_btn.value:
        file_info = upload_btn.value[0]
        uploaded_image['data'] = file_info['content']
        with output:
            output.clear_output()
            print(f"Gambar '{file_info['name']}' berhasil diunggah.")

# Proses gambar & OCR
def process_image(btn):
    if uploaded_image['data'] is None:
        with output:
            output.clear_output()
            print("Silakan unggah plat nomor kendaraan.")
        return

    image_stream = io.BytesIO(uploaded_image['data'])
    pil_image = Image.open(image_stream).convert('RGB')
    image_np = np.array(pil_image)
    
    # Menyesuaikan kontras gambar berdasarkan slider
    image_rgb = adjust_contrast(image_np, contrast_slider.value)

    # OCR
    results = reader.readtext(image_rgb)

    if not results:
        with output:
            output.clear_output()
            print("Tidak ada teks terdeteksi.")
        return

    # Gabungkan teks berdasarkan baris
    lines = []  # Setiap elemen = [(bbox, text, y_center)]
    for bbox, text, prob in results:
        yc = y_center(bbox)
        placed = False
        for line in lines:
            if abs(yc - line[0][2]) < 25:
                line.append((bbox, text, yc))
                placed = True
                break
        if not placed:
            lines.append([(bbox, text, yc)])

    # Urutkan tiap baris berdasarkan posisi X, lalu gabungkan teksnya
    lines_sorted = []
    for line in lines:
        sorted_line = sorted(line, key=lambda x: x[0][0][0])  # sort by x
        texts = [text for _, text, _ in sorted_line]
        lines_sorted.append(" ".join(texts))

    # Gambar kotak dan teks di atas gambar
    for bbox, text, prob in results:
        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))
        cv2.rectangle(image_rgb, top_left, bottom_right, (0, 255, 0), 2)
        cv2.putText(image_rgb, text, top_left, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # Tampilkan hasil
    with output:
        output.clear_output()
        plt.figure(figsize=(12, 8))
        plt.imshow(image_rgb)
        plt.axis('off')
        plt.title('Deteksi Plat Nomor Kendaraan', fontsize=16, color='navy')

        hasil = "\n".join(lines_sorted)
        plt.figtext(0.5, 0.01, f'Hasil OCR:\n{hasil}', wrap=True, horizontalalignment='center', fontsize=14, color='green')
        plt.show()

# Reset handler
def reset_app(btn):
    uploaded_image['data'] = None
    with output:
        output.clear_output()
        print("Aplikasi di-reset, silakan unggah gambar baru.")

# Download handler
def download_ocr(btn):
    if uploaded_image['data'] is None:
        with output:
            output.clear_output()
            print("Silakan unggah gambar dan deteksi plat nomor terlebih dahulu.")
        return

    # Menggabungkan hasil OCR untuk diunduh
    image_stream = io.BytesIO(uploaded_image['data'])
    pil_image = Image.open(image_stream).convert('RGB')
    image_np = np.array(pil_image)
    
    # Menyesuaikan kontras gambar berdasarkan slider
    image_rgb = adjust_contrast(image_np, contrast_slider.value)

    # OCR
    results = reader.readtext(image_rgb)

    lines = []  # Setiap elemen = [(bbox, text, y_center)]
    for bbox, text, prob in results:
        yc = y_center(bbox)
        placed = False
        for line in lines:
            if abs(yc - line[0][2]) < 25:
                line.append((bbox, text, yc))
                placed = True
                break
        if not placed:
            lines.append([(bbox, text, yc)])

    # Gabungkan hasil OCR
    lines_sorted = []
    for line in lines:
        sorted_line = sorted(line, key=lambda x: x[0][0][0])  # sort by x
        texts = [text for _, text, _ in sorted_line]
        lines_sorted.append(" ".join(texts))

    hasil = "\n".join(lines_sorted)

    # Simpan hasil OCR dalam file teks
    with open("hasil_ocr.txt", "w") as f:
        f.write(hasil)

    with output:
        output.clear_output()
        print("Hasil OCR telah disimpan sebagai 'hasil_ocr.txt'")

# Menampilkan gambar asli
def show_original_image(btn):
    if uploaded_image['data'] is None:
        with output:
            output.clear_output()
            print("Silakan unggah gambar terlebih dahulu.")
        return

    image_stream = io.BytesIO(uploaded_image['data'])
    pil_image = Image.open(image_stream).convert('RGB')
    
    with output:
        output.clear_output()
        plt.figure(figsize=(8, 6))
        plt.imshow(pil_image)
        plt.axis('off')
        plt.title('Gambar Asli')
        plt.show()

# Hubungkan tombol dan tampilkan
upload_btn.observe(on_upload_change, names='value')
process_btn.on_click(process_image)
reset_btn.on_click(reset_app)
download_btn.on_click(download_ocr)
show_original_btn.on_click(show_original_image)

# Tambahkan instruksi dengan font besar dan warna yang menarik
instructions = Label(value="Unggah gambar plat nomor kendaraan untuk mendeteksi teks. Setelah gambar diunggah, klik tombol 'Deteksi Plat Nomor'.", 
                     style={'description_width': 'initial'}, layout={'width': '100%'})

# Ubah tampilan background dan padding untuk elemen GUI
layout = VBox([instructions, 
               HBox([upload_btn, process_btn, reset_btn, download_btn, show_original_btn], layout={'justify_content': 'center', 'align_items': 'center'}),
               contrast_slider,
               output], layout={'padding': '20px', 'background_color': '#f0f8ff', 'width': '80%'})

# Display tampilan
display(layout)


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
